In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

C:\Users\dheer\anaconda3\envs\spark311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [10]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Group by

In [3]:
emp_data = [
    (1,'manish',50000,'IT','m'),
    (2,'vikash',60000,'sales','m'),
    (3,'raushan',70000,'marketing','m'),
    (4,'mukesh',80000,'IT','m'),
    (5,'priti',90000,'sales','f'),
    (6,'nikita',45000,'marketing','f'),
    (7,'ragini',55000,'marketing','f'),
    (8,'rashi',100000,'IT','f'),
    (9,'aditya',65000,'IT','m'),
    (10,'rahul',50000,'marketing','m'),
    (11,'rakhi',50000,'IT','f'),
    (12,'akhilesh',90000,'sales','m')
]

schema = ['emp_id', 'emp_name', 'salary', 'department', 'gender']

emp_df = spark.createDataFrame(emp_data, schema)

emp_df.show()

+------+--------+------+----------+------+
|emp_id|emp_name|salary|department|gender|
+------+--------+------+----------+------+
|     1|  manish| 50000|        IT|     m|
|     2|  vikash| 60000|     sales|     m|
|     3| raushan| 70000| marketing|     m|
|     4|  mukesh| 80000|        IT|     m|
|     5|   priti| 90000|     sales|     f|
|     6|  nikita| 45000| marketing|     f|
|     7|  ragini| 55000| marketing|     f|
|     8|   rashi|100000|        IT|     f|
|     9|  aditya| 65000|        IT|     m|
|    10|   rahul| 50000| marketing|     m|
|    11|   rakhi| 50000|        IT|     f|
|    12|akhilesh| 90000|     sales|     m|
+------+--------+------+----------+------+



In [5]:
emp_df.groupBy('department')\
      .agg(sum('salary')).show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        IT|     345000|
|     sales|     240000|
| marketing|     220000|
+----------+-----------+



In [13]:
window = Window.partitionBy('department', 'gender').orderBy('salary')
emp_df.withColumn('row_number', row_number().over(window))\
      .show(truncate=False)

+------+--------+------+----------+------+----------+
|emp_id|emp_name|salary|department|gender|row_number|
+------+--------+------+----------+------+----------+
|11    |rakhi   |50000 |IT        |f     |1         |
|8     |rashi   |100000|IT        |f     |2         |
|1     |manish  |50000 |IT        |m     |1         |
|9     |aditya  |65000 |IT        |m     |2         |
|4     |mukesh  |80000 |IT        |m     |3         |
|6     |nikita  |45000 |marketing |f     |1         |
|7     |ragini  |55000 |marketing |f     |2         |
|10    |rahul   |50000 |marketing |m     |1         |
|3     |raushan |70000 |marketing |m     |2         |
|5     |priti   |90000 |sales     |f     |1         |
|2     |vikash  |60000 |sales     |m     |1         |
|12    |akhilesh|90000 |sales     |m     |2         |
+------+--------+------+----------+------+----------+



In [15]:
window = Window.partitionBy('department').orderBy('salary')
emp_df.withColumn('row_number', row_number().over(window))\
      .withColumn('Rank', rank().over(window))\
       .withColumn('Dense rank', dense_rank().over(window))\
      .show(truncate=False)

+------+--------+------+----------+------+----------+----+----------+
|emp_id|emp_name|salary|department|gender|row_number|Rank|Dense rank|
+------+--------+------+----------+------+----------+----+----------+
|1     |manish  |50000 |IT        |m     |1         |1   |1         |
|11    |rakhi   |50000 |IT        |f     |2         |1   |1         |
|9     |aditya  |65000 |IT        |m     |3         |3   |2         |
|4     |mukesh  |80000 |IT        |m     |4         |4   |3         |
|8     |rashi   |100000|IT        |f     |5         |5   |4         |
|6     |nikita  |45000 |marketing |f     |1         |1   |1         |
|10    |rahul   |50000 |marketing |m     |2         |2   |2         |
|7     |ragini  |55000 |marketing |f     |3         |3   |3         |
|3     |raushan |70000 |marketing |m     |4         |4   |4         |
|2     |vikash  |60000 |sales     |m     |1         |1   |1         |
|5     |priti   |90000 |sales     |f     |2         |2   |2         |
|12    |akhilesh|900

In [17]:
window = Window.partitionBy('department', 'gender').orderBy('salary')
emp_df.withColumn('row_number', row_number().over(window))\
      .withColumn('Rank', rank().over(window))\
       .withColumn('Dense rank', dense_rank().over(window))\
      .show(truncate=False)

+------+--------+------+----------+------+----------+----+----------+
|emp_id|emp_name|salary|department|gender|row_number|Rank|Dense rank|
+------+--------+------+----------+------+----------+----+----------+
|11    |rakhi   |50000 |IT        |f     |1         |1   |1         |
|8     |rashi   |100000|IT        |f     |2         |2   |2         |
|1     |manish  |50000 |IT        |m     |1         |1   |1         |
|9     |aditya  |65000 |IT        |m     |2         |2   |2         |
|4     |mukesh  |80000 |IT        |m     |3         |3   |3         |
|6     |nikita  |45000 |marketing |f     |1         |1   |1         |
|7     |ragini  |55000 |marketing |f     |2         |2   |2         |
|10    |rahul   |50000 |marketing |m     |1         |1   |1         |
|3     |raushan |70000 |marketing |m     |2         |2   |2         |
|5     |priti   |90000 |sales     |f     |1         |1   |1         |
|2     |vikash  |60000 |sales     |m     |1         |1   |1         |
|12    |akhilesh|900